# Homework: Linear regression with penguins

In this exercise, we investigate the relationship between **flipper length** and **bill depth**.

We will first fit a simple linear regression model, then consider whether knowing a penguin's species changes the prediction.

**Learning goals**
- Fit and interpret a simple linear regression.
- Extract the slope, its standard error, and the coefficient of determination ($R^2$).
- Use a fitted model to make a prediction.
- Explain how an additional variable (species) can change a prediction.


## 1. Load and inspect the data

We use the Palmer Penguins dataset, available through the `seaborn` package.

If necessary, install the packages by running `!pip install seaborn statsmodels` in a separate cell.


In [1]:
import seaborn as sns
import pandas as pd
import statsmodels.formula.api as smf

penguins = sns.load_dataset("penguins")

# Keep the variables needed for this exercise and remove rows with missing values.
data = penguins[["bill_depth_mm", "flipper_length_mm", "species"]].dropna()

data.head()

,bill_depth_mm,flipper_length_mm,species
0,18.7,181.0,Adelie
1,17.4,186.0,Adelie
2,18.0,195.0,Adelie
4,19.3,193.0,Adelie
5,20.6,190.0,Adelie


How many observations remain after removing missing values?

In [2]:
# Your code here
data.count()

bill_depth_mm        342
flipper_length_mm    342
species              342
dtype: int64

In [3]:
data = data.rename(columns={"bill_depth_mm": "bill", "flipper_length_mm":"flipper"})

## 2. Fit a simple linear regression

We want to explain **bill depth** using **flipper length**.

The model is

$$
\text{bill depth}_i = \beta_0 + \beta_1\,\text{flipper length}_i + \varepsilon_i.
$$

Fit the model using ordinary least squares (OLS).


In [4]:
# Fit the regression model
mdl = smf.ols('bill ~ flipper', data = data).fit()
# Display the regression results
mdl.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   bill   R-squared:                       0.341
Model:                            OLS   Adj. R-squared:                  0.339
Method:                 Least Squares   F-statistic:                     175.8
Date:                Fri, 13 Nov 2026   Prob (F-statistic):           1.23e-32
Time:                        12:44:13   Log-Likelihood:                -646.21
No. Observations:                 342   AIC:                             1296.
Df Residuals:                     340   BIC:                             1304.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     33.6251      1.245     27.000      0.000      31.176      36.075
flipper       -0.0820      0.006    -13.261      0.000      -0.094      -0.070
==============================================================================
Omnibus:                       11.696   Durbin-Watson:                   2.249
Prob(Omnibus):                  0.003   Jarque-Bera (JB):                8.177
Skew:                           0.254   Prob(JB):                       0.0168
Kurtosis:                       2.439   Cond. No.                     2.89e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.89e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## 3. Questions

Use the regression output to answer the following questions.

1. What is the estimated **slope**? Interpret it in the context of the penguins.
2. What is the **standard error of the slope**?
3. What is the model's **$R^2$** value? Explain briefly what it tells you.
4. Use the fitted regression line to predict the bill depth (in mm) of a penguin with a flipper length of **225 mm**.



In [5]:
#1. What is the estimated **slope**? Interpret it in the context of the penguins.
slope=mdl.params
print("Slope is " + str(round(slope["flipper"],4))+ " which suggests a very week negative relationship.")
#2. What is the **standard error of the slope**?
mdl.bse["flipper"]
#3. What is the model's **$R^2$** value? Explain briefly what it tells you.
mdl.rsquared
print("The model givess 34% better results than if we just predict by the average.")
#4. Use the fitted regression line to predict the bill depth (in mm) of a penguin with a flipper length of **225 mm**.
pred = pd.DataFrame({"flipper":[225]})
mdl.predict(pred)


Slope is -0.082 which suggests a very week negative relationship.
The model givess 34% better results than if we just predict by the average.


0    15.176345
dtype: float64

## 4. Does species matter?



In [10]:
mdl_spec = smf.ols("bill ~ flipper + species + 0 ", data = data).fit()

#NOT GOOD -> ONLY ONE SLOPE!!!
mdl_spec = smf.ols("bill ~ flipper * species ", data = data).fit()
mdl_spec.summary()



<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   bill   R-squared:                       0.762
Model:                            OLS   Adj. R-squared:                  0.758
Method:                 Least Squares   F-statistic:                     214.6
Date:                Fri, 13 Nov 2026   Prob (F-statistic):          2.86e-102
Time:                        13:08:06   Log-Likelihood:                -472.38
No. Observations:                 342   AIC:                             956.8
Df Residuals:                     336   BIC:                             979.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                        7.4749      2.306      3.242      0.001       2.940      12.010
species[T.Chinstrap]            -7.1403      3.994     -1.788      0.075     -14.996       0.716
species[T.Gentoo]              -15.7118      3.742     -4.199      0.000     -23.072      -8.351
flipper                          0.0572      0.012      4.718      0.000       0.033       0.081
flipper:species[T.Chinstrap]     0.0351      0.021      1.706      0.089      -0.005       0.076
flipper:species[T.Gentoo]        0.0497      0.018      2.730      0.007       0.014       0.085
==============================================================================
Omnibus:                        3.517   Durbin-Watson:                   2.200
Prob(Omnibus):                  0.172   Jarque-Bera (JB):                3.224
Skew:                           0.214   Prob(JB):                        0.200
Kurtosis:                       3.207   Cond. No.                     2.07e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.07e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""